# 🤖 Agente de Código Full Stack

Agente que recibe instrucciones en lenguaje natural y genera una **app web completa en Next.js**.

**Stack:**
- **GitHub Models** (`gpt-4o`) como LLM
- **E2B** como sandbox seguro para escribir y compilar código
- **Runtime Summary** para no quedarse sin contexto en tareas largas

**Credenciales necesarias (Colab Secrets 🔑):**
- `GITHUB_TOKEN` → https://github.com/settings/tokens
- `E2B_API_KEY`  → https://e2b.dev


## 1. Instalación

In [ ]:
!pip install openai e2b-code-interpreter -q


## 2. Credenciales

In [ ]:
import os
from google.colab import userdata

os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')
os.environ['E2B_API_KEY']  = userdata.get('E2B_API_KEY')

print('✅ Credenciales cargadas')


## 3. Clonar / montar el repositorio

Si estás en Colab, clona el repo. Si ya tenés el código en el entorno, saltá esta celda.


In [ ]:
# Opción A: clonar desde GitHub
!git clone https://github.com/mijaelryan/fullstack-agent.git
%cd fullstack-agent

# Opción B: ya estás en el directorio correcto
# import os
# print('Directorio actual:', os.getcwd())


## 4. Inicializar cliente y sandbox

In [ ]:
from agent import build_client, build_sandbox

client = build_client()   # usa os.environ['GITHUB_TOKEN']
sbx    = build_sandbox()  # usa os.environ['E2B_API_KEY']


## 5. Verificación rápida de herramientas (Parte 1)

Antes de correr el agente, comprobamos que las herramientas del sandbox funcionan.


In [ ]:
from lib.sbx_tools import list_directory, write_file, read_file, search_file_content

# Escribir un archivo de prueba
r = write_file(sbx, '/home/user/test.txt', 'Hola desde sbx_tools!\nLínea 2')
print('write_file:', r)

# Leerlo
r = read_file(sbx, '/home/user/test.txt')
print('read_file:', r)

# Listar directorio
r = list_directory(sbx, '/home/user')
print('list_directory:', r)

# Buscar patrón
r = search_file_content(sbx, 'Hola', path='/home/user')
print('search_file_content:', r)


## 6. Verificación de compresión de contexto (Parte 2)

Simulamos un historial que supere el límite de `6,000` tokens estimados
para verificar que `maybe_compress` actúa correctamente.
El historial de prueba se mantiene por debajo de `8,000` tokens
para no superar el límite del modelo de resumen (`gpt-4o-mini`).


In [ ]:
from lib.context import count_tokens, maybe_compress, MAX_TOKENS

# Crear historial falso que supere el límite de 6k tokens
# pero no el límite del modelo de resumen (8k)
fake_messages = [
    {"role": "user",      "content": [{"text": "x" * 500}]},
    {"role": "assistant", "content": [{"text": "y" * 500}]},
] * 30  # ~7,500 tokens estimados — supera 6k pero no 8k

tokens_antes = count_tokens(fake_messages)
print(f'Tokens estimados: {tokens_antes:,}  |  Límite: {MAX_TOKENS:,}')
assert tokens_antes > MAX_TOKENS, 'El historial debe superar el límite'

compressed = maybe_compress(fake_messages, client)
print(f'Mensajes después de comprimir: {len(compressed)} (era {len(fake_messages)})')
assert len(compressed) < len(fake_messages), 'La compresión debe reducir los mensajes'
print('✅ Compresión de contexto OK')


## 7. Parte 4 — Probar el agente

### Tarea 1: Crear la app


In [ ]:
from agent import run_agent

historial = []

# Tarea 1
historial, respuesta = run_agent(
    'Crea una app de lista de tareas estilo Windows 95.',
    client=client,
    sbx=sbx,
    messages=historial,
)

print('\n--- RESPUESTA FINAL ---')
print(respuesta)


### Tarea 2: Ajuste sobre el mismo historial


In [ ]:
# Tarea 2 — mismo historial, el agente recuerda el contexto anterior
historial, respuesta = run_agent(
    'Los íconos del nav son blancos y no se ven. Arreglalo.',
    client=client,
    sbx=sbx,
    messages=historial,
)

print('\n--- RESPUESTA FINAL ---')
print(respuesta)


## 8. Cerrar el sandbox


In [ ]:
sbx.kill()
print('✅ Sandbox cerrado')


---
## Resumen de arquitectura

| Componente | Archivo | Rol |
|---|---|---|
| Herramientas filesystem | `lib/sbx_tools.py` | list, read, write, search, replace, glob, execute_bash |
| Compresión de contexto | `lib/context.py` | Runtime Summary a 6k tokens |
| System prompt | `lib/prompts.py` | Instrucciones Next.js al agente |
| Loop del agente | `agent.py` | `run_agent()` con tool calls |
| Demo | `notebook.ipynb` | Este archivo |

**Flujo:**
```
run_agent(query)
  → maybe_compress()   # comprime si >6k tokens estimados
  → llm()              # GitHub Models gpt-4o
  → execute_tool()     # sbx_tools en E2B
  → _try_download()    # descarga workspace/ al sistema local
  → loop hasta sin tool calls
```

> **Nota técnica:** El agente usa `gpt-4o` para generar código y `gpt-4o-mini`
> para comprimir el historial — optimización intencional de tokens.
